# 第 1 课：文档抽取（结构化输出）

把一段发票文字交给大模型，拿回一个**经过校验的 Python 对象**。

**这一课学到的**
- LLM：一次调用的结构、JSON 输出、校验失败后带错误重试、用准确率评估效果
- Python：`dict` / `list`、f-string、类型标注、Pydantic、`try/except`、`pathlib`

| C# 里你熟悉的 | 这一课的 Python 写法 |
|---|---|
| `IChatClient` / `HttpClient` | `OpenAI(...)` 客户端 |
| `record` + DataAnnotations / FluentValidation | Pydantic `BaseModel` + `Field(...)` |
| `JsonSerializer.Deserialize<T>()` | `Invoice.model_validate_json(...)` |
| `try { } catch (JsonException)` | `try: ... except ValidationError:` |

> 用法：从上往下，每个格子按 `Shift + Enter` 运行，运行完看输出再往下。

## 1. 连接大模型

GLM、DeepSeek、Moonshot 都兼容 OpenAI 的接口，所以统一用 `openai` 这个包，换模型只改 `.env` 里的三个变量。

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # 读取同目录的 .env 文件，相当于 appsettings.json + 环境变量

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.getenv("LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/"),
)
MODEL = os.getenv("LLM_MODEL", "glm-4-flash")
print("使用模型:", MODEL)

## 2. 最简单的一次调用

`messages` 是一个 **list，里面每一项是 dict**，相当于 C# 的 `List<Dictionary<string, object>>`。Python 里经常直接用 dict 表示 JSON，不用先定义类。

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,                 # Python 的“命名参数”，和 C# 的 model: MODEL 一样
    messages=[
        {"role": "system", "content": "你是一个简洁的助手。"},
        {"role": "user", "content": "用一句话解释什么是发票。"},
    ],
)

print(resp.choices[0].message.content)
print(f"token：输入 {resp.usage.prompt_tokens}，输出 {resp.usage.completion_tokens}")  # f-string = C# 的 $"..."

## 3. 定义 DTO：Pydantic 就是 record + 校验

- `str | None` 相当于 C# 的 `string?`
- `Field(gt=0)` 相当于 `[Range]`，`pattern=` 相当于 `[RegularExpression]`
- 字段名用 `snake_case`（下划线），这是 Python 的命名习惯，C# 是 `PascalCase`

In [ ]:
from datetime import date
from decimal import Decimal
from pydantic import BaseModel, Field, ValidationError


class Invoice(BaseModel):
    invoice_number: str = Field(description="发票号码")
    issue_date: date | None = Field(default=None, description="开票日期，yyyy-MM-dd")
    seller_name: str | None = Field(default=None, description="销售方名称")
    buyer_name: str | None = Field(default=None, description="购买方名称")
    total_amount: Decimal | None = Field(default=None, gt=0, description="价税合计金额")
    currency: str | None = Field(default=None, pattern=r"^[A-Z]{3}$", description="币种代码，如 CNY、USD")


# 好玩的一点：Pydantic 能直接生成 JSON Schema，等下原样塞进提示词里告诉模型要什么格式
import json
print(json.dumps(Invoice.model_json_schema(), ensure_ascii=False, indent=2))

先手动试一下校验：故意给一个错误的日期和金额，看 Pydantic 怎么报错。

In [ ]:
try:
    Invoice.model_validate({"invoice_number": "A001", "issue_date": "2026/10/08", "total_amount": -5})
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], err["msg"])

## 4. 核心：抽取 + 校验 + 带错误重试

流程：调用模型 → 用 Pydantic 校验 → 不通过就把**错误信息追加进对话**，让模型自己改，最多 3 次。

这就是 Agent 循环的雏形：**模型的输出是不可信输入，必须校验**。

In [ ]:
SYSTEM_PROMPT = f"""你是发票信息抽取助手。只输出一个 JSON 对象，不要输出其他文字。
JSON 必须符合这个 Schema：
{json.dumps(Invoice.model_json_schema(), ensure_ascii=False)}
文档里找不到的字段填 null，不要猜。"""


def strip_code_fence(text: str) -> str:
    """有些模型会把 JSON 包在 ```json ... ``` 里，去掉它。"""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return text.strip()


def extract(document: str, max_attempts: int = 3) -> tuple[Invoice | None, int]:
    """返回 (抽取结果, 尝试次数)。tuple 相当于 C# 的 (Invoice?, int) 元组。"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": document},
    ]

    for attempt in range(1, max_attempts + 1):   # range(1, 4) 就是 1, 2, 3
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0,                            # 抽取任务要稳定
            response_format={"type": "json_object"},  # 要求输出合法 JSON
        )
        raw = resp.choices[0].message.content or ""

        try:
            return Invoice.model_validate_json(strip_code_fence(raw)), attempt
        except ValidationError as e:
            errors = "；".join(f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors())
            print(f"  第 {attempt} 次校验失败：{errors}")
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": f"你的输出有问题：{errors}。请修正后重新输出完整 JSON。"})

    return None, max_attempts

## 5. 跑样本，算每个字段的准确率

`samples/` 里每份 `.txt` 都有一个同名 `.expected.json` 作为标准答案。往里面加你自己的真实样本就能扩大评估集。

In [ ]:
from collections import Counter
from pathlib import Path

correct, total = Counter(), Counter()   # Counter 是一个“默认值为 0 的字典”

for txt in sorted(Path("samples").glob("*.txt")):
    invoice, attempts = extract(txt.read_text(encoding="utf-8"))
    if invoice is None:
        print(f"❌ {txt.name} 抽取失败")
        continue
    print(f"✅ {txt.name}（第 {attempts} 次成功）")
    print("  ", invoice.model_dump(mode="json"))

    expected_file = txt.with_suffix(".expected.json")
    if expected_file.exists():
        expected = Invoice.model_validate_json(expected_file.read_text(encoding="utf-8"))
        for field in Invoice.model_fields:            # 遍历所有字段名，类似反射
            total[field] += 1
            if getattr(invoice, field) == getattr(expected, field):
                correct[field] += 1
            else:
                print(f"   ⚠ {field} 不一致：{getattr(invoice, field)!r} ≠ {getattr(expected, field)!r}")

print("\n各字段准确率：")
for field in total:
    print(f"  {field:<15} {correct[field]}/{total[field]}  {correct[field] / total[field]:.0%}")

## 动手练习

1. 往 `samples/` 放 10 份你手头的真实发票或合同文字（加上 `.expected.json`），看准确率。
2. 给 `Invoice` 加一个明细字段 `items: list[InvoiceItem]`，再定义一个 `InvoiceItem` 模型，看模型在列表上的表现。
3. 把 `.env` 里的模型换成更强或更便宜的，对比准确率和 token 花费。
4. 读 PDF：`uv add pypdf`，用 `PdfReader(path).pages[0].extract_text()` 拿到文字再交给 `extract()`。